# 2. Experiments

Recompute both tests for the four classifiers, six vocabulary selectors and three
corpora from the prepared matrices. Run all cells from the supplement folder.
Six CSV tables are written to `results/` for notebook 3 and the paper's summaries.
The full run replaces the supplied reference tables only after every corpus finishes.


## Protocol
Five outer folds; ten A/B training pairs with approximately 10% of sample A replaced; 500-word
vocabularies for Test 1. Test 2 compares 500-word profiles with 400- and 600-word profiles.
Vocabulary selection and population z-score statistics use training rows only.
Held-out texts are shared between compared fits. BA is balanced accuracy from
out-of-fold predictions. RBO_EXT uses depth 50 and persistence 0.95.

Models: D = L1 Delta centroid; C = cosine centroid; L = balanced logistic regression;
S = balanced linear SVM, all using the original objectives and C=1 for L/S.
The archived large runs used GPU FP64 optimizers. This portable implementation uses
scikit-learn on CPU; numerical optimization can change last digits and near-tied ranks.

A profile averages absolute, per-text L1-normalized feature contributions to the
true-author versus best-rival margin. It is a model contribution profile.
Scalar intercepts are excluded from feature rankings. Ties use lexicographic order.
Related observations stay in the same outer fold. RusLit9 has 259 texts in 254 groups.
IMDb62/C20 is unavailable because fewer than 600 eligible features remain.


## Setup

In [ ]:
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.svm import LinearSVC
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
CORPORA = ("IMDb62", "FreDraCor", "RusLit9")
SELECTORS = ("MFW", "EDR", "ZMAD", "GMDZ", "C20", "DF")
METHODS = ("D", "C", "L", "S")
SIZES = (400, 500, 600)
BASE = 500
FOLDS, REPEATS = 5, 10
TOP, P = 50, 0.95
SEED = 20261019
N_JOBS = 4


class Fit(NamedTuple):
    prediction: np.ndarray
    top: np.ndarray
    selected: np.ndarray
    profiles: np.ndarray | None
    mass: np.ndarray | None


## Training-only vocabulary selection
MFW: total counts; EDR: mean(TF)^2 / SD(TF); ZMAD: mean absolute z-score;
GMDZ: mean absolute pairwise z-score difference; C20: MFW with document frequency
at least 20%; DF: document frequency. Zero-variance words are excluded. Score ties
use descending total count, then lexicographic word order. An insufficient vocabulary
is recorded as unavailable, without relaxing its rule. Craig-document Zeta is excluded.

In [ ]:
def select_words(tf, counts, words):
    tf = tf.tocsc()
    text_count, word_count = tf.shape
    document_frequency = np.diff(tf.indptr)
    # In CSC format, nonzero values are stored together by word.
    columns = np.repeat(np.arange(word_count), document_frequency)
    mean = np.asarray(tf.mean(axis=0)).ravel()
    mean_square = np.asarray(tf.power(2).mean(axis=0)).ravel()
    variance = np.maximum(mean_square - mean * mean, 0)
    has_variance = variance > 10 * np.finfo(float).eps * mean_square
    std = np.sqrt(variance)
    std[~has_variance] = 1
    total_count = np.asarray(counts.sum(axis=0)).ravel().astype(float)
    deviations = np.abs(tf.data - mean[columns])
    mean_absolute_deviation = np.bincount(
        columns, weights=deviations, minlength=word_count
    )
    zero_counts = text_count - document_frequency
    mean_absolute_deviation += zero_counts * mean
    mean_absolute_deviation /= text_count * std
    # Sorted nonzero values and implicit zeros give pairwise differences without a dense matrix.
    order = np.lexsort((tf.data, columns))
    rank = np.arange(tf.nnz) - np.repeat(tf.indptr[:-1], document_frequency)
    rank += np.repeat(text_count - document_frequency + 1, document_frequency)
    weights = (2 * rank - text_count - 1) * tf.data[order]
    pair_difference_sum = np.bincount(columns, weights=weights, minlength=word_count)
    scores = {
        "MFW": total_count,
        "EDR": mean * mean / std,
        "ZMAD": mean_absolute_deviation,
        "GMDZ": 2 * pair_difference_sum / (text_count * (text_count - 1) * std),
        "C20": total_count,
        "DF": document_frequency.astype(float),
    }
    # Break equal scores by total count, then alphabetically.
    rankings = {}
    for name in SELECTORS:
        eligible = (
            has_variance & (document_frequency >= 0.2 * text_count)
            if name == "C20" else has_variance
        )
        indices = np.flatnonzero(eligible)
        order = np.lexsort((words[indices], -total_count[indices], -scores[name][indices]))
        rankings[name] = indices[order][:max(SIZES)]
    return mean, std, rankings

## Paired training samples and rank comparison

In [ ]:
def paired_training(pool, labels, rng):
    sample_size = int(0.9 * len(pool))
    swap_count = max(1, round(0.1 * sample_size))
    # Keep at least two texts per author in the initial training sample.
    protected = np.concatenate([rng.choice(pool[labels[pool] == author], 2, replace=False)
                                for author in np.unique(labels)])
    reserve = rng.choice(np.setdiff1d(pool, protected), len(pool) - sample_size, replace=False)
    left = np.setdiff1d(pool, reserve)
    incoming = rng.choice(reserve, swap_count, replace=False)
    # Replace texts within the same author to preserve author counts.
    outgoing = []
    for author in np.unique(labels[incoming]):
        author_texts = left[labels[left] == author]
        count = np.sum(labels[incoming] == author)
        outgoing.append(rng.choice(author_texts, count, replace=False))
    outgoing = np.concatenate(outgoing)
    right = np.sort(np.r_[np.setdiff1d(left, outgoing), incoming])
    return left, right


def rbo(left, right):
    seen_left, seen_right, value = set(), set(), 0.0
    for depth, (a, b) in enumerate(zip(left, right), 1):
        seen_left.add(int(a))
        seen_right.add(int(b))
        agreement = len(seen_left & seen_right) / depth
        value += (1 - P) * P ** (depth - 1) * agreement
    return value + P ** len(left) * agreement


## Fit the four models and extract author profiles

In [ ]:
def evaluate(x, labels, query, true_labels, selected_words):
    classes = np.arange(labels.max() + 1)
    centroids = np.stack([x[labels == author].mean(axis=0) for author in classes])
    centroid_norm = np.linalg.norm(centroids, axis=1)
    query_norm = np.linalg.norm(query, axis=1)
    if np.any(centroid_norm == 0) or np.any(query_norm == 0):
        raise ValueError("Cosine model encountered a zero vector")
    normalized_query, normalized_centroids = query / query_norm[:,
        None], centroids / centroid_norm[:, None]
    predictions, top_words, full_profiles = [], [], []
    for method in METHODS:
        if method == "D":
            scores = -cdist(query, centroids, metric="cityblock") / x.shape[1]
        elif method == "C":
            scores = normalized_query @ normalized_centroids.T - 1
        else:
            if method == "L":
                model = LogisticRegression(C=1, class_weight="balanced", solver="lbfgs", tol=1e-4,
                                           max_iter=50000, random_state=SEED)
            else:
                model = LinearSVC(C=1, class_weight="balanced", dual=False, tol=1e-6,
                                  max_iter=50000, random_state=SEED)
            model.fit(x, labels)
            if np.max(model.n_iter_) >= 50000:
                raise RuntimeError(f"{method} reached the iteration limit")
            coefficients, intercept = model.coef_, model.intercept_
            if len(classes) == 2:
                coefficients, intercept = np.r_[-coefficients / 2,
                    coefficients / 2], np.r_[-intercept / 2, intercept / 2]
            scores = query @ coefficients.T + intercept
        # Compare the true author with the highest-scoring alternative.
        masked = scores.copy()
        masked[np.arange(len(query)), true_labels] = -np.inf
        rival = masked.argmax(axis=1)
        if method == "D":
            contributions = (
                np.abs(query - centroids[rival])
                - np.abs(query - centroids[true_labels])
            ) / x.shape[1]
        elif method == "C":
            contributions = normalized_query * (
                normalized_centroids[true_labels] - normalized_centroids[rival]
            )
        else:
            contributions = query * (coefficients[true_labels] - coefficients[rival])
        # Normalize each text before averaging its contributions by author.
        magnitude = np.abs(contributions)
        if np.any(magnitude.sum(axis=1) == 0):
            raise ValueError("A text has no nonzero feature contributions")
        shares = magnitude / magnitude.sum(axis=1, keepdims=True)
        profiles = np.stack([shares[true_labels == author].mean(axis=0) for author in classes])
        if np.any((profiles > 0).sum(axis=1) < TOP):
            raise ValueError("An author profile has fewer than TOP nonzero words")
        predictions.append(scores.argmax(axis=1))
        top_words.append(np.stack([np.lexsort((selected_words,
            -profile))[:TOP] for profile in profiles]))

        full_profiles.append(profiles)

    return (np.array(predictions, dtype=np.int32), np.stack(top_words),
            np.stack(full_profiles))

## Aggregate contribution profiles

Rank displacement is conditional on shared lemmas and uses their original positions.
Contribution shares are normalized within each text and averaged by true author.
The summary tables compare model/selector ranges and top-50 contribution mass.


In [ ]:
def rank_shift_totals(words, left_words, left_profiles, right_words, right_profiles):
    """Compare shared words in their original rankings and bin by reference position."""
    _, left_idx, right_idx = np.intersect1d(
        left_words, right_words, assume_unique=True, return_indices=True
    )
    left_ranks = rankdata(-left_profiles, axis=-1)
    right_ranks = rankdata(-right_profiles, axis=-1)
    order = np.lexsort((np.broadcast_to(words[left_words], left_profiles.shape), -left_profiles))
    shifts = np.full(left_profiles.shape, np.nan)
    shifts[..., left_idx] = np.abs(left_ranks[..., left_idx] - right_ranks[..., right_idx])
    ordered = np.take_along_axis(shifts, order, axis=-1)
    return np.nansum(ordered, axis=1), np.isfinite(ordered).sum(axis=1)


def manuscript_profiles(corpus, mass, mass_fits, shift_sum, shift_count, shift_fits, authors):
    expected_fits = np.array([[2 * REPEATS * FOLDS], [FOLDS]])
    complete = mass_fits == expected_fits
    mean_mass = mass / np.where(complete, mass_fits * authors, np.nan)[..., None, None]
    top50 = pd.DataFrame([
        dict(corpus=corpus, test=test, selector=selector, method=method,
             top50_percent=100 * mean_mass[t, s, m, :TOP].sum())
        for t, test in enumerate(("test1", "test2"))
        for s, selector in enumerate(SELECTORS) if complete[t, s]
        for m, method in enumerate(METHODS)
    ])

    mfw_mass = mean_mass[:, SELECTORS.index("MFW")].mean(axis=0)
    np.testing.assert_allclose(mfw_mass.sum(axis=1), 1, atol=1e-12)
    np.testing.assert_array_less(np.diff(mfw_mass, axis=1), 1e-14)
    shares = pd.DataFrame([
        dict(corpus=corpus, model=method, depth=position, share_percent=100 * value)
        for method, curve in zip(METHODS, mfw_mass)
        for position, value in enumerate(curve, 1)
    ])

    np.testing.assert_array_equal(shift_fits, [FOLDS * REPEATS, FOLDS, FOLDS])
    if np.any(shift_count == 0):
        raise ValueError(f"{corpus}: an MFW position has no matched word in a comparison")
    displacement = shift_sum / shift_count
    coverage = shift_count / (shift_fits * authors)[:, None, None]
    shifts = pd.DataFrame([
        dict(corpus=corpus, test=test, method=method, position=position,
             mean_abs_shift=value, coverage=observed)
        for test, curves, fractions in (
            ("test1", displacement[0], coverage[0]),
            ("test2", displacement[1:].mean(axis=0), coverage[1:].mean(axis=0)),
        )
        for method, curve, fraction in zip(METHODS, curves, fractions)
        for position, (value, observed) in enumerate(zip(curve, fraction), 1)
    ])
    return {
        "MFW_rank_displacements.csv": shifts,
        "MFW_contribution_shares.csv": shares,
        "contribution_top50.csv": top50,
    }


def manuscript_summaries(tables):
    metrics = tables["metrics.csv"].query('status == "complete"')
    rows = []
    for corpus, data in metrics.groupby("corpus", sort=False):
        for test in ("test1", "test2"):
            rbo = data.pivot(index="method", columns="selector", values=f"rbo_{test}")
            rows.append(dict(
                corpus=corpus, test=test,
                mean_range_across_models=(rbo.max(axis=0) - rbo.min(axis=0)).mean(),
                mean_range_across_selectors=(rbo.max(axis=1) - rbo.min(axis=1)).mean(),
            ))
    spread = tables["contribution_top50.csv"].groupby(
        ["corpus", "test", "method"], sort=False
    ).top50_percent.agg(dictionaries="count", min_top50_percent="min", max_top50_percent="max")
    spread["top50_span_pp"] = spread.max_top50_percent - spread.min_top50_percent
    return {
        "rbo_spread.csv": pd.DataFrame(rows),
        "contribution_dictionary_spread.csv": spread.reset_index(),
    }


## Run both tests and aggregate out-of-fold results

In [ ]:
def experiment(corpus):
    folder = ROOT / "data/preprocessed" / corpus
    counts = sparse.load_npz(folder / "counts.npz").astype(float)
    with np.load(folder / "metadata.npz", allow_pickle=False) as meta:
        labels = meta["labels"]
        words = meta["words"]
        groups = meta["groups"]
        lengths = meta["token_count"]
    tf = counts.multiply(1 / lengths[:, None]).tocsr()
    authors, n = labels.max() + 1, len(labels)
    selectors, methods = len(SELECTORS), len(METHODS)
    mass = np.zeros((2, selectors, methods, BASE))
    mass_fits = np.zeros((2, selectors), dtype=int)
    shift_sum = np.zeros((3, methods, BASE))
    shift_count = np.zeros((3, methods, BASE), dtype=int)
    shift_fits = np.zeros(3, dtype=int)

    def add_shift(comparison, left, right):
        total, observed = rank_shift_totals(
            words, left.selected, left.profiles, right.selected, right.profiles
        )
        shift_sum[comparison] += total
        shift_count[comparison] += observed
        shift_fits[comparison] += 1

    def add_mass(test, selector_index, fit):
        mass[test, selector_index] += fit.mass
        mass_fits[test, selector_index] += 1
    # Store out-of-fold predictions and per-author profile comparisons.
    prediction1 = np.full((REPEATS, 2, selectors, methods, n), -1, dtype=np.int32)
    prediction2 = np.full((len(SIZES), selectors, methods, n), -1, dtype=np.int32)
    stability1 = np.full((FOLDS, REPEATS, selectors, methods, authors), np.nan)
    stability2 = np.full((FOLDS, 2, selectors, methods, authors), np.nan)

    def fit_set(train, test, sizes):
        # Vocabulary and standardization are fitted only on the training rows.
        mean, std, rankings = select_words(tf[train], counts[train], words)
        results = {}
        for selector in SELECTORS:
            for size in sizes:
                if len(rankings[selector]) < size:
                    continue
                chosen = rankings[selector][:size]
                x = (tf[train][:, chosen].toarray() - mean[chosen]) / std[chosen]
                query = (tf[test][:, chosen].toarray() - mean[chosen]) / std[chosen]
                mfw = selector == "MFW"
                prediction, top, profiles = evaluate(
                    x, labels[train], query, labels[test], words[chosen]
                )
                rank_mass = np.sort(profiles, axis=-1)[..., ::-1].sum(axis=1) if size == BASE else None
                results[selector, size] = Fit(
                    prediction, chosen[top], chosen, profiles if mfw else None, rank_mass
                )
        return results

    # Keep related or duplicate texts in the same fold.
    grouped = len(np.unique(groups)) < n
    if grouped:
        splitter = StratifiedGroupKFold(FOLDS, shuffle=True, random_state=SEED)
        folds = splitter.split(np.zeros(n), labels, groups)
    else:
        splitter = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)
        folds = splitter.split(np.zeros(n), labels)
    with threadpool_limits(1):
        for fold, (pool, test) in enumerate(folds):
            assert not set(groups[pool]) & set(groups[test])
            assert len(np.unique(labels[pool])) == len(np.unique(labels[test])) == authors
            pairs = []
            for repeat in range(REPEATS):
                seed = np.random.SeedSequence([SEED, fold, repeat])
                rng = np.random.default_rng(seed)
                pairs.append(paired_training(pool, labels, rng))
            tasks = [(train, (BASE,)) for pair in pairs for train in pair] + [(pool, SIZES)]
            fitted = Parallel(n_jobs=N_JOBS, prefer="threads")(
                delayed(fit_set)(train, test, sizes) for train, sizes in tasks)
            for si, selector in enumerate(SELECTORS):
                for repeat in range(REPEATS):
                    left = fitted[2 * repeat].get((selector, BASE))
                    right = fitted[2 * repeat + 1].get((selector, BASE))
                    for side, result in enumerate((left, right)):
                        if result is not None:
                            prediction1[repeat, side, si][:, test] = result.prediction
                            add_mass(0, si, result)
                    if left is not None and right is not None:
                        for mi in range(methods):
                            stability1[fold, repeat, si, mi] = [
                                rbo(a, b) for a, b in zip(left.top[mi], right.top[mi])
                            ]
                        if selector == "MFW":
                            add_shift(0, left, right)
                by_size = [fitted[-1].get((selector, size)) for size in SIZES]
                for size_index, result in enumerate(by_size):
                    if result is not None:
                        size = SIZES[size_index]
                        prediction2[size_index, si][:, test] = result.prediction
                        if size == BASE:
                            add_mass(1, si, result)
                if all(result is not None for result in by_size):
                    baseline = by_size[SIZES.index(BASE)]
                    for comparison, size_index in enumerate((0, len(SIZES) - 1)):
                        target = by_size[size_index]
                        for mi in range(methods):
                            stability2[fold, comparison, si, mi] = [
                                rbo(a, b) for a, b in zip(baseline.top[mi], target.top[mi])
                            ]
                        if selector == "MFW":
                            add_shift(comparison + 1, baseline, target)
            print(f"{corpus}: fold {fold + 1}/{FOLDS} finished", flush=True)

    # Aggregate held-out predictions and stability scores into one row per configuration.
    rows = []
    for si, selector in enumerate(SELECTORS):
        for mi, method in enumerate(METHODS):
            p1, p2 = prediction1[:, :, si, mi], prediction2[:, si, mi]
            complete1, complete2 = np.all(p1 >= 0), np.all(p2 >= 0)
            ba1 = []
            if complete1:
                for repeat in p1:
                    scores = [balanced_accuracy_score(labels, side) for side in repeat]
                    ba1.append(np.mean(scores))
            rb1 = stability1[:, :, si, mi].mean(axis=(0, 2)) if complete1 else []
            row = dict(corpus=corpus, selector=selector, method=method,
                       ba_test1=np.mean(ba1) if complete1 else np.nan,
                       ba_test1_sd=np.std(ba1, ddof=1) if complete1 and REPEATS > 1 else np.nan,
                       rbo_test1=np.mean(rb1) if complete1 else np.nan,
                       rbo_test1_sd=np.std(rb1, ddof=1) if complete1 and REPEATS > 1 else np.nan,
                       prediction_change=np.mean(p1[:, 0] != p1[:, 1]) if complete1 else np.nan,
                       ba_test2_400=balanced_accuracy_score(labels, p2[0])
                           if np.all(p2[0] >= 0) else np.nan,
                       ba_test2=balanced_accuracy_score(labels, p2[SIZES.index(BASE)])
                           if np.all(p2[SIZES.index(BASE)] >= 0) else np.nan,
                       ba_test2_600=balanced_accuracy_score(labels, p2[-1])
                           if np.all(p2[-1] >= 0) else np.nan,
                       rbo_test2=np.mean(stability2[:, :, si, mi]) if complete2 else np.nan,
                       rbo_test2_400=np.mean(stability2[:, 0, si, mi]) if complete2 else np.nan,
                       rbo_test2_600=np.mean(stability2[:, 1, si, mi]) if complete2 else np.nan,
                       status="complete" if complete1 and complete2 else "unavailable_features",
                       base_size=BASE, low_size=SIZES[0], high_size=SIZES[-1], folds=FOLDS,
                       repeats=REPEATS, top=TOP, rbo_p=P, documents=n, authors=authors,
                       engine="scikit-learn CPU")
            rows.append(row)
    tables = manuscript_profiles(corpus, mass, mass_fits, shift_sum, shift_count, shift_fits, authors)
    tables["metrics.csv"] = pd.DataFrame(rows)
    return tables

## Export the six result tables

All selections, standardization and fitting use training rows only.

In [ ]:
tables = {}
for corpus in CORPORA:
    for name, table in experiment(corpus).items():
        tables.setdefault(name, []).append(table)
combined = {name: pd.concat(parts, ignore_index=True) for name, parts in tables.items()}
combined.update(manuscript_summaries(combined))
output = ROOT / "results"
output.mkdir(exist_ok=True)
for name, table in combined.items():
    table.to_csv(output / name, index=False)
print(f"Saved {len(combined)} result tables.")
display(combined["rbo_spread.csv"])
display(combined["contribution_dictionary_spread.csv"])


Continue with [03_figures.ipynb](03_figures.ipynb).